# Analyst Agent — Demo Notebook

This notebook demonstrates the agent running an end-to-end analysis on the
sample employee salary dataset.

**Prerequisites:**
1. `pip install -e '.[dev]'` from the `agent/` directory
2. Copy `.env.example` → `.env` and add your `ANTHROPIC_API_KEY`
3. Run `python data/sample_datasets/generate_samples.py` to create datasets

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from src.logging_config import configure_logging
configure_logging('INFO')

## Step 1: Profile the dataset

In [ ]:
from src.ingestion import DataLoader, DataProfiler, QualityScorer

loader = DataLoader()
profiler = DataProfiler()
scorer = QualityScorer()

df, source = loader.load('../data/sample_datasets/employee_salary.csv')
schema = scorer.score(profiler.profile(df, source))

print(schema.summary_for_llm())

In [ ]:
from IPython.display import Markdown
Markdown(scorer.report(schema))

## Step 2: Run autonomous analysis

In [ ]:
from src.engine import AnalystAgent
from src.memory import MemoryStore

memory = MemoryStore()  # in-memory for demo
agent = AnalystAgent(memory=memory)

question = "What factors most strongly predict employee salary, and are there identifiable salary clusters?"

for event in agent.analyse(
    source='../data/sample_datasets/employee_salary.csv',
    question=question,
):
    print(event)

## Step 3: Generate the report

In [ ]:
from src.reporting import ReportGenerator
from pathlib import Path

session = agent.last_session()

gen = ReportGenerator(output_dir=Path('../workspace/reports'))
html_path, md_path = gen.generate(session, schema)

print(f'HTML report: {html_path}')
print(f'MD report:   {md_path}')

In [ ]:
from IPython.display import IFrame
IFrame(str(html_path), width='100%', height=800)

## Step 4: Follow-up question (L4 dialectic)

In [ ]:
for response in agent.chat("Is the salary difference between Engineering and Sales statistically significant?"):
    print(response)